# 03 — Rollout tracking

A standalone `RolloutTracker` collects ordinary MDP/POMDP progress and completed episode statistics. The caller owns the tracker and updates it once for every output, starting with the initial reset. Environments do not store statistics.

`tracker.observation`, `step_index`, `episode_index`, `cumulative_reward`, and `action_requested` describe the current rollout. `tracker.episodes` is a tuple of immutable `EpisodeSummary` records with `episode_index`, `reward`, `length`, `terminated`, and `truncated`.


In [ ]:
from mouse_gym import EnvConfig, RolloutTracker, make_env, make_group_env

env = make_env(EnvConfig(id="CartPole-v1", seed=0, seed_mode="constant"))
tracker = RolloutTracker()
for _ in range(1000):
    action = env.action_space.sample() if tracker.action_requested else None
    output = env.step(action)
    tracker.update(output=output)

rewards = [episode.reward for episode in tracker.episodes]
lengths = [episode.length for episode in tracker.episodes]
print(f"Completed episodes: {len(rewards)}")
print(f"Mean return: {sum(rewards) / len(rewards):.1f}")
print(f"Mean length: {sum(lengths) / len(lengths):.1f}")


## Clear completed summaries

`clear()` removes completed summaries but preserves the current output, running reward, counters, and boundary flags. An episode already in progress therefore retains its full return when it later completes.


In [ ]:
tracker.clear()
assert tracker.episodes == ()
for _ in range(500):
    action = env.action_space.sample() if tracker.action_requested else None
    tracker.update(output=env.step(action))
print(tracker.episodes[:3])
env.close()


## Multiple environments

Keep one tracker for each env and pair trackers with outputs in env order. The trackers work identically with sequential or threaded stepping.


In [ ]:
env = make_group_env([
    EnvConfig(id="CartPole-v1", seed=0, seed_mode="constant", name="CartPole-0"),
    EnvConfig(id="MountainCar-v0", seed=1, seed_mode="constant", name="MountainCar-0"),
])
trackers = [RolloutTracker() for _ in env.envs]
try:
    for _ in range(500):
        actions = [single.action_space.sample() if tracker.action_requested else None
                   for single, tracker in zip(env.envs, trackers)]
        for tracker, output in zip(trackers, env.step(actions)):
            tracker.update(output=output)
    for name, tracker in zip(env.names, trackers):
        rewards = [episode.reward for episode in tracker.episodes]
        mean = sum(rewards) / len(rewards) if rewards else float("nan")
        print(name, len(rewards), mean)
finally:
    env.close()


## Reward transforms

`reward_transform` runs before the returned output is collected. Summaries include transformed rewards, including reset frames. The callable receives `step_index`, `state`, `action`, `reward`, `terminated`, `truncated`, and `next_state`. A reset frame has `step_index == 0`, `reward == 0.0`, `action is None`, and false termination flags.


In [ ]:
import gymnasium as gym

class OneStepEnv(gym.Env):
    observation_space = gym.spaces.Discrete(2)
    action_space = gym.spaces.Discrete(2)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        return 0, {}

    def step(self, action):
        return 1, 1.0, True, False, {}


def reward_transform(*, step_index, reward, **_kwargs):
    return -0.1 if step_index == 0 else reward * 10.0


env = make_env(EnvConfig(env_fn=OneStepEnv, reward_transform=reward_transform))
tracker = RolloutTracker()
try:
    for _ in range(4):
        action = env.action_space.sample() if tracker.action_requested else None
        tracker.update(output=env.step(action))
    print(tracker.episodes)  # two episodes, each with reward 9.9 and length 1
finally:
    env.close()


## Repeated tasks

Use `RepeatTaskTracker` for `RepeatTaskEnv`: `episodes` holds inner episode summaries and `tasks` holds outer task summaries. `repeat_step_index` and `repeat_cumulative_reward` describe the inner episode. `step_index` and `cumulative_reward` describe the task. `action_requested` is false between episodes and tasks; pass `None` to advance a reset. See [04 — Repeat a task](04_repeat_task_env.ipynb).
